# Bank Account Fraud Detection - Federated Learning

## Continuação do SOTA Benchmark com Aprendizado Federado

Este notebook mantém 100% do preprocessamento, feature engineering e estrutura do notebook original `bank_account_fraud_sota_benchmark`, adicionando **Federated Learning** usando o framework **Flower** com estratégias **Bagging** e **Cyclic**.

### Modelos Federalizados:
- XGBoost
- LightGBM
- CatBoost

### Configuração FL:
- **3 clientes** com quantidades balanceadas de fraude/não-fraude
- **Estratégias**: Bagging (paralelo) e Cyclic (sequencial)
- **Dataset**: BAF Base
- **Validação**: Temporal (Meses 0-5: Treino | Mês 6: Validação | Mês 7: Teste)

---
## Seção 1: Imports e Configuração

In [ ]:
# Computação e Dados
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Utilitários
import warnings
import pickle
import json
from typing import Tuple, List, Dict, Any, Optional
from collections import defaultdict
import time

# Pré-processamento
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Métricas
from sklearn.metrics import (
    roc_auc_score, 
    roc_curve, 
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Modelos de Árvore
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool

# Federated Learning com Flower
import flwr as fl
from flwr.common import (
    Status,
    Code,
    FitIns,
    FitRes,
    EvaluateIns,
    EvaluateRes,
    Parameters,
    Scalar,
    NDArrays,
    parameters_to_ndarrays,
    ndarrays_to_parameters
)
from flwr.server.strategy import FedAvg
from flwr.simulation import start_simulation

# Configurações
warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Configuração de visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print(" Imports concluídos com sucesso!")
print(f"Versão Flower: {fl.__version__}")
print(f"Versão XGBoost: {xgb.__version__}")
print(f"Versão LightGBM: {lgb.__version__}")

---
## Seção 2: Carregamento e Pré-processamento

### Mantém 100% do preprocessamento original:
1. Conversão de -1 para NaN
2. Remoção de ruído (Hard Drop)
3. Imputação com mediana
4. Feature Engineering
5. One-Hot Encoding

In [ ]:
# Caminho do dataset
DATA_PATH = r"c:\Users\candi\OneDrive\Desktop\Federated-Learning\baf_data_&_code\Base.csv"

# Carregar dataset
print(" Carregando dataset BAF Base...")
df = pd.read_csv(DATA_PATH)
print(f"Shape original: {df.shape}")
print(f"Fraudes: {df['fraud_bool'].sum()} ({df['fraud_bool'].mean()*100:.2f}%)")

# Visualização inicial
df.head()

In [ ]:
# PASSO 1: Converter -1 para NaN (conforme especificação do dataset)
print("\n PASSO 1: Convertendo -1 para NaN...")
df.replace(-1, np.nan, inplace=True)
print(f"Valores NaN por coluna:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

In [ ]:
# PASSO 2: Remoção de Ruído (Hard Drop)
print("\n PASSO 2: Remoção de Ruído...")

# 2.1 Remover colunas com >20% nulos
null_threshold = 0.2
null_cols = df.columns[df.isnull().mean() > null_threshold]
print(f"Removendo colunas com >{null_threshold*100}% nulos: {list(null_cols)}")
df.drop(columns=null_cols, inplace=True)

# 2.2 Remover colunas com variância zero
zero_var_cols = df.columns[df.nunique() == 1]
print(f"Removendo colunas com variância zero: {list(zero_var_cols)}")
df.drop(columns=zero_var_cols, inplace=True)

# 2.3 Remover linhas com device_distinct_emails_8w NaN (se existir)
if 'device_distinct_emails_8w' in df.columns:
    before = len(df)
    df.dropna(subset=['device_distinct_emails_8w'], inplace=True)
    print(f"Removidas {before - len(df)} linhas com device_distinct_emails_8w NaN")

# 2.4 Remover velocity_24h (alta correlação com velocity_4w)
if 'velocity_24h' in df.columns:
    print(f"Removendo velocity_24h (correlação 0.89 com velocity_4w)")
    df.drop(columns=['velocity_24h'], inplace=True)

print(f"\nShape após remoção de ruído: {df.shape}")

In [ ]:
# PASSO 3: Imputação com Mediana
print("\n PASSO 3: Imputação de valores faltantes com mediana...")

# Identificar colunas numéricas com nulos
numeric_cols = df.select_dtypes(include=[np.number]).columns
null_numeric_cols = [col for col in numeric_cols if df[col].isnull().any()]

print(f"Colunas numéricas com nulos: {null_numeric_cols}")

# Imputar com mediana
for col in null_numeric_cols:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)
    print(f"  {col}: imputado com {median_val}")

print(f"\nTotal de NaN restantes: {df.isnull().sum().sum()}")

---
## Seção 3: Feature Engineering

### Features adicionadas:
1. **Frequências**: `device_os_frequency`, `source_frequency`
2. **Ratios**: `velocity_ratio_6h_4w`
3. **Z-scores mensais normalizados**: velocidades, zip_count, bank_branch_count

In [ ]:
print("\n Feature Engineering Avançado...")

# 1. Frequências
if 'device_os' in df.columns:
    df['device_os_frequency'] = df.groupby('device_os')['device_os'].transform('count')
    print(" device_os_frequency criada")

if 'source' in df.columns:
    df['source_frequency'] = df.groupby('source')['source'].transform('count')
    print(" source_frequency criada")

# 2. Ratio de velocidades (captura bursts)
if 'velocity_6h' in df.columns and 'velocity_4w' in df.columns:
    df['velocity_ratio_6h_4w'] = df['velocity_6h'] / (df['velocity_4w'] + 1)  # +1 para evitar divisão por zero
    print(" velocity_ratio_6h_4w criada")

# 3. Z-scores mensais normalizados
if 'month' in df.columns:
    velocity_cols = ['velocity_6h', 'velocity_4w', 'zip_count_4w', 'bank_branch_count_8w']
    
    for col in velocity_cols:
        if col in df.columns:
            zscore_col = f'{col}_zscore_monthly'
            df[zscore_col] = df.groupby('month')[col].transform(
                lambda x: (x - x.mean()) / (x.std() + 1e-6)
            )
            print(f" {zscore_col} criada")

print(f"\nShape final após feature engineering: {df.shape}")

# ============================================================================
# FAIRNESS FEATURE ENGINEERING
# ============================================================================
print("\nCriando features de interação para Fairness...")

# 2.3.1 Income x Age interaction
df['income_x_age'] = df['income'] * df['customer_age']

# 2.3.2 Age group binning + Income deviation from age group mean
df['age_group'] = pd.cut(df['customer_age'],
                         bins=[0, 30, 50, 100],
                         labels=['young', 'middle', 'senior'])
income_by_age = df.groupby('age_group')['income'].transform('mean')
df['income_vs_age_group_mean'] = df['income'] - income_by_age

# 2.3.3 Employment status + Income interaction (concatenação categórica)
df['employment_income_cat'] = df['employment_status'] + '_' + \
                               pd.qcut(df['income'], q=4, labels=['Q1','Q2','Q3','Q4']).astype(str)

# 2.3.4 Customer Age x Credit Risk Score
df['age_x_credit_risk'] = df['customer_age'] * df['credit_risk_score']

# 2.3.5 Age above 50 flag (para análise de fairness)
df['age_above_50'] = (df['customer_age'] > 50).astype(int)

# 2.3.6 Income per year of age
df['income_per_age'] = df['income'] / (df['customer_age'] + 1)

# 2.3.7 Credit risk normalizado por employment status
emp_credit_mean = df.groupby('employment_status')['credit_risk_score'].transform('mean')
emp_credit_std = df.groupby('employment_status')['credit_risk_score'].transform('std').replace(0, 1)
df['credit_risk_vs_employment'] = (df['credit_risk_score'] - emp_credit_mean) / emp_credit_std

print("Features de fairness criadas:")
print("  - income_x_age")
print("  - age_group")
print("  - income_vs_age_group_mean")
print("  - employment_income_cat")
print("  - age_x_credit_risk")
print("  - age_above_50")
print("  - income_per_age")
print("  - credit_risk_vs_employment")


# ============================================================================
# ADDITIONAL FEATURE ENGINEERING
# ============================================================================

print("\nCriando features adicionais...")

# 2.4.1 Proporção de validações telefônicas
df['phone_validation_score'] = df['phone_home_valid'] + df['phone_mobile_valid']

# 2.4.2 Session length anomaly (comparado com média do source)
source_session_mean = df.groupby('source')['session_length_in_minutes'].transform('mean')
df['session_length_vs_source'] = df['session_length_in_minutes'] - source_session_mean

# 2.4.3 Email similarity x Email free (interação interessante)
df['email_similarity_x_free'] = df['name_email_similarity'] * df['email_is_free']

# 2.4.4 Bank months vs Address months ratio
df['bank_vs_address_months'] = df['bank_months_count'] / (df['current_address_months_count'] + 1)

# 2.4.5 Days since request log transform (para reduzir skewness)
df['days_since_request_log'] = np.log1p(df['days_since_request'])

# 2.4.6 Proposed credit limit buckets
df['credit_limit_bucket'] = pd.cut(df['proposed_credit_limit'],
                                   bins=[0, 200, 500, 1000, 2000, np.inf],
                                   labels=['very_low', 'low', 'medium', 'high', 'very_high'])

print("Features adicionais criadas:")
print("  - phone_validation_score")
print("  - session_length_vs_source")
print("  - email_similarity_x_free")
print("  - bank_vs_address_months")
print("  - days_since_request_log")
print("  - credit_limit_bucket")


In [ ]:
# Identificar tipos de features
print("\nIdentificando tipos de features...")

# Features categóricas base
categorical_features = []
for col in df.columns:
    if df[col].dtype == 'object' or (df[col].dtype in ['int64', 'float64'] and df[col].nunique() < 10):
        if col not in ['fraud_bool', 'month']:
            categorical_features.append(col)

# Adicionar features categóricas criadas manualmente
categorical_features_manual = ['age_group', 'employment_income_cat', 'credit_limit_bucket']
for col in categorical_features_manual:
    if col in df.columns and col not in categorical_features:
        categorical_features.append(col)

# Converter para string (necessário para encoding)
for col in categorical_features:
    df[col] = df[col].astype(str)

# Features numéricas (excluindo target, month e categóricas)
numeric_features = [col for col in df.select_dtypes(include=[np.number]).columns
                   if col not in ['fraud_bool', 'month'] and col not in categorical_features]

print(f"Features categóricas ({len(categorical_features)}): {categorical_features}")
print(f"Features numéricas ({len(numeric_features)}): {numeric_features[:10]}... (showing first 10)")


---
## Seção 4: Validação Temporal

### Split:
- **Treino**: Meses 0-5
- **Validação**: Mês 6 (para Optuna)
- **Teste**: Mês 7 (holdout final)

In [ ]:
print("\nSplit Temporal...")

# Separar por mês
train_df = df[df['month'] <= 5].copy()
val_df = df[df['month'] == 6].copy()
test_df = df[df['month'] == 7].copy()

print(f"Treino (meses 0-5): {len(train_df)} amostras | {train_df['fraud_bool'].mean()*100:.2f}% fraudes")
print(f"Validação (mês 6): {len(val_df)} amostras | {val_df['fraud_bool'].mean()*100:.2f}% fraudes")
print(f"Teste (mês 7): {len(test_df)} amostras | {test_df['fraud_bool'].mean()*100:.2f}% fraudes")

# IMPORTANTE: Guardar age_above_50 ANTES de separar features
# Esta coluna será usada APENAS para análise de fairness, NÃO para treino
print("\nGuardando age_above_50 para análise de fairness...")
age_above_50_train_full = train_df['age_above_50'].copy()
age_above_50_val_full = val_df['age_above_50'].copy()
age_above_50_test_full = test_df['age_above_50'].copy()
print("age_above_50 salvo para treino, validação e teste")

# Separar features e target (EXCLUINDO age_above_50)
feature_cols = [c for c in df.columns if c not in ['fraud_bool', 'month', 'age_above_50']]

X_train = train_df[feature_cols]
y_train = train_df['fraud_bool']

X_val = val_df[feature_cols]
y_val = val_df['fraud_bool']

X_test = test_df[feature_cols]
y_test = test_df['fraud_bool']

print(f"\nFeatures shape: {X_train.shape[1]} (age_above_50 EXCLUÍDA do treino)")


In [ ]:
# One-Hot Encoding das features categóricas
print("\n One-Hot Encoding...")

if len(categorical_features) > 0:
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    
    # Fit no treino
    encoder.fit(X_train[categorical_features])
    
    # Transform em treino, validação e teste
    X_train_cat = encoder.transform(X_train[categorical_features])
    X_val_cat = encoder.transform(X_val[categorical_features])
    X_test_cat = encoder.transform(X_test[categorical_features])
    
    # Criar nomes de colunas
    cat_feature_names = encoder.get_feature_names_out(categorical_features)
    
    # Combinar features numéricas e categóricas
    X_train_final = np.hstack([
        X_train[numeric_features].values,
        X_train_cat
    ])
    
    X_val_final = np.hstack([
        X_val[numeric_features].values,
        X_val_cat
    ])
    
    X_test_final = np.hstack([
        X_test[numeric_features].values,
        X_test_cat
    ])
    
    # Criar lista de nomes de features
    all_feature_names = list(numeric_features) + list(cat_feature_names)
    
    print(f" Encoding concluído: {X_train_final.shape[1]} features totais")
else:
    X_train_final = X_train[numeric_features].values
    X_val_final = X_val[numeric_features].values
    X_test_final = X_test[numeric_features].values
    all_feature_names = numeric_features
    print(f" Sem features categóricas: {len(numeric_features)} features numéricas")

---
## Seção 5: Métricas Customizadas

### Métrica Principal: TPR @ 5% FPR
- TPR (True Positive Rate / Recall): % de fraudes corretamente identificadas
- FPR (False Positive Rate): % de clientes legítimos marcados como fraude
- **Meta do Benchmark**: TPR > 0.52 @ FPR = 5%

In [ ]:
def get_threshold_at_fpr(y_true: np.ndarray, y_prob: np.ndarray, fpr_target: float = 0.05) -> float:
    """
    Retorna o threshold que resulta em FPR <= fpr_target.
    
    Args:
        y_true: Labels verdadeiros (0 ou 1)
        y_prob: Probabilidades preditas (0 a 1)
        fpr_target: FPR alvo (padrão: 0.05 = 5%)
    
    Returns:
        threshold: Valor de corte ótimo
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    valid_indices = np.where(fpr <= fpr_target)[0]
    
    if len(valid_indices) == 0:
        return 1.0  # threshold máximo se nenhum ponto satisfaz
    
    best_idx = valid_indices[np.argmax(tpr[valid_indices])]
    return thresholds[best_idx]


def calc_tpr_at_fpr(y_true: np.ndarray, y_prob: np.ndarray, fpr_target: float = 0.05) -> float:
    """
    Calcula o TPR máximo mantendo FPR <= fpr_target.
    
    Args:
        y_true: Labels verdadeiros (0 ou 1)
        y_prob: Probabilidades preditas (0 a 1)
        fpr_target: FPR alvo (padrão: 0.05 = 5%)
    
    Returns:
        tpr: True Positive Rate no threshold ótimo
    """
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    valid_indices = np.where(fpr <= fpr_target)[0]
    
    if len(valid_indices) == 0:
        return 0.0
    
    best_idx = valid_indices[np.argmax(tpr[valid_indices])]
    return tpr[best_idx]


def evaluate_model(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    """
    Avalia um modelo com métricas completas.
    
    Args:
        y_true: Labels verdadeiros
        y_prob: Probabilidades preditas
    
    Returns:
        dict: Dicionário com métricas
    """
    # Threshold para FPR = 5%
    threshold = get_threshold_at_fpr(y_true, y_prob, fpr_target=0.05)
    y_pred = (y_prob >= threshold).astype(int)
    
    metrics = {
        'roc_auc': roc_auc_score(y_true, y_prob),
        'tpr_at_5fpr': calc_tpr_at_fpr(y_true, y_prob, fpr_target=0.05),
        'threshold': threshold,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0)
    }
    
    return metrics


print(" Funções de métricas definidas!")

---
## Seção 5.5: Otimização de Hiperparâmetros com Optuna

Antes de executar o treinamento federalizado, vamos otimizar os hiperparâmetros de cada modelo usando o framework **Optuna**.

### Configuração da Otimização:
- **Framework**: Optuna (TPE Sampler)
- **Métrica de Otimização**: TPR @ 5% FPR (maximização)
- **Conjunto de Validação**: Mês 6
- **Número de Trials**: 50 por modelo
- **Modelos**: XGBoost, LightGBM, CatBoost

Os hiperparâmetros otimizados serão usados no treinamento federalizado, com exceção de `n_estimators`/`iterations`, que serão substituídos por `NUM_LOCAL_ROUNDS` (20 boosting rounds por cliente).

In [ ]:
import optuna
from optuna.samplers import TPESampler

# Configurar Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(" Optuna importado e configurado!")
print(f"Versão Optuna: {optuna.__version__}")

### 5.5.1 Otimização XGBoost

Otimizando hiperparâmetros do XGBoost para maximizar TPR @ 5% FPR no conjunto de validação.

In [ ]:
def objective_xgb(trial):
    """Função objetivo para XGBoost."""
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'hist',
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 2),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2),
        'random_state': RANDOM_STATE
    }
    
    model = xgb.XGBClassifier(**params)
    model.fit(X_train_final, y_train.values, verbose=False)
    
    y_prob = model.predict_proba(X_val_final)[:, 1]
    return calc_tpr_at_fpr(y_val.values, y_prob, fpr_target=0.05)

print(" Iniciando otimização XGBoost com Optuna...")
print("Métrica: TPR @ 5% FPR (maximização)")
print("Validação: Mês 6")
print("-" * 60)

study_xgb = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=RANDOM_STATE)
)

study_xgb.optimize(objective_xgb, n_trials=50, show_progress_bar=True)

print(f"\n Melhor TPR @ 5% FPR (XGBoost): {study_xgb.best_value:.4f}")
print(f"\nMelhores hiperparâmetros XGBoost:")
for key, value in study_xgb.best_params.items():
    print(f"  {key}: {value}")

# Salvar melhores params para uso no FL
xgb_best_params = study_xgb.best_params.copy()
xgb_best_params['objective'] = 'binary:logistic'
xgb_best_params['eval_metric'] = 'auc'
xgb_best_params['tree_method'] = 'hist'
xgb_best_params['random_state'] = RANDOM_STATE

### 5.5.2 Otimização LightGBM

Otimizando hiperparâmetros do LightGBM para maximizar TPR @ 5% FPR no conjunto de validação.

In [ ]:
def objective_lgbm(trial):
    """Função objetivo para LightGBM."""
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 2),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2),
        'verbose': -1,
        'random_state': RANDOM_STATE
    }
    
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train_final, y_train.values)
    
    y_prob = model.predict_proba(X_val_final)[:, 1]
    return calc_tpr_at_fpr(y_val.values, y_prob, fpr_target=0.05)

print(" Iniciando otimização LightGBM com Optuna...")
print("Métrica: TPR @ 5% FPR (maximização)")
print("Validação: Mês 6")
print("-" * 60)

study_lgbm = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=RANDOM_STATE)
)

study_lgbm.optimize(objective_lgbm, n_trials=50, show_progress_bar=True)

print(f"\n Melhor TPR @ 5% FPR (LightGBM): {study_lgbm.best_value:.4f}")
print(f"\nMelhores hiperparâmetros LightGBM:")
for key, value in study_lgbm.best_params.items():
    print(f"  {key}: {value}")

# Salvar melhores params para uso no FL
lgb_best_params = study_lgbm.best_params.copy()
lgb_best_params['objective'] = 'binary'
lgb_best_params['metric'] = 'auc'
lgb_best_params['boosting_type'] = 'gbdt'
lgb_best_params['verbose'] = -1
lgb_best_params['random_state'] = RANDOM_STATE

### 5.5.3 Otimização CatBoost

Otimizando hiperparâmetros do CatBoost para maximizar TPR @ 5% FPR no conjunto de validação.

In [ ]:
def objective_catboost(trial):
    """Função objetivo para CatBoost."""
    params = {
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'iterations': trial.suggest_int('iterations', 100, 500),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_strength': trial.suggest_float('random_strength', 0, 10),
        'verbose': False,
        'random_seed': RANDOM_STATE
    }
    
    model = CatBoostClassifier(**params)
    model.fit(X_train_final, y_train.values, verbose=False)
    
    y_prob = model.predict_proba(X_val_final)[:, 1]
    return calc_tpr_at_fpr(y_val.values, y_prob, fpr_target=0.05)

print(" Iniciando otimização CatBoost com Optuna...")
print("Métrica: TPR @ 5% FPR (maximização)")
print("Validação: Mês 6")
print("-" * 60)

study_catboost = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=RANDOM_STATE)
)

study_catboost.optimize(objective_catboost, n_trials=50, show_progress_bar=True)

print(f"\n Melhor TPR @ 5% FPR (CatBoost): {study_catboost.best_value:.4f}")
print(f"\nMelhores hiperparâmetros CatBoost:")
for key, value in study_catboost.best_params.items():
    print(f"  {key}: {value}")

# Salvar melhores params para uso no FL
catboost_best_params = study_catboost.best_params.copy()
catboost_best_params['loss_function'] = 'Logloss'
catboost_best_params['eval_metric'] = 'AUC'
catboost_best_params['verbose'] = False
catboost_best_params['random_seed'] = RANDOM_STATE

### Resumo da Otimização

Os hiperparâmetros otimizados acima serão usados no treinamento federalizado. Cada modelo foi otimizado para maximizar **TPR @ 5% FPR** usando o conjunto de validação (Mês 6).

**Importante**: No FL, vamos adaptar os parâmetros de boosting rounds:
- `n_estimators` / `iterations` → serão substituídos por `NUM_LOCAL_ROUNDS` (20 rounds por cliente)
- Os demais hiperparâmetros serão mantidos conforme otimizado pelo Optuna

### Comparação dos Resultados:

| Modelo | TPR @ 5% FPR (Validação) |
|--------|-------------------------|
| XGBoost | *(ver output acima)* |
| LightGBM | *(ver output acima)* |
| CatBoost | *(ver output acima)* |

---
---
# PARTE FEDERALIZADA
---
---

## Seção 6: Particionamento Federado

### Estratégia de Particionamento:
- **3 clientes** com quantidades **balanceadas** de fraude/não-fraude
- Cada cliente recebe aproximadamente:
  - 1/3 das fraudes
  - 1/3 dos não-fraudulentos
- Particionamento **estratificado** para manter distribuição de classes
- Mantém split temporal (treino/validação/teste)

In [ ]:
print("\n Particionamento Federado para 3 Clientes...\n")

NUM_CLIENTS = 3

def create_federated_partitions(
    X: np.ndarray, 
    y: np.ndarray, 
    num_clients: int = 3
) -> Dict[int, Tuple[np.ndarray, np.ndarray]]:
    """
    Particiona dados de forma balanceada entre clientes.
    
    Args:
        X: Features
        y: Labels (0 ou 1)
        num_clients: Número de clientes
    
    Returns:
        dict: {client_id: (X_client, y_client)}
    """
    # Separar fraudes e não-fraudes
    fraud_idx = np.where(y == 1)[0]
    non_fraud_idx = np.where(y == 0)[0]
    
    # Embaralhar índices
    np.random.shuffle(fraud_idx)
    np.random.shuffle(non_fraud_idx)
    
    # Dividir em chunks
    fraud_chunks = np.array_split(fraud_idx, num_clients)
    non_fraud_chunks = np.array_split(non_fraud_idx, num_clients)
    
    # Criar partições
    partitions = {}
    for i in range(num_clients):
        # Combinar fraudes e não-fraudes
        client_idx = np.concatenate([fraud_chunks[i], non_fraud_chunks[i]])
        
        # Embaralhar para misturar fraudes e não-fraudes
        np.random.shuffle(client_idx)
        
        # Extrair dados
        X_client = X[client_idx]
        y_client = y.iloc[client_idx].values if hasattr(y, 'iloc') else y[client_idx]
        
        partitions[i] = (X_client, y_client)
        
        # Estatísticas
        fraud_count = np.sum(y_client == 1)
        total = len(y_client)
        print(f"Cliente {i}: {total} amostras | {fraud_count} fraudes ({fraud_count/total*100:.2f}%)")
    
    return partitions


# Criar partições para treino, validação e teste
print(" Particionando dados de TREINO:")
train_partitions = create_federated_partitions(X_train_final, y_train, NUM_CLIENTS)

print("\n Particionando dados de VALIDAÇÃO:")
val_partitions = create_federated_partitions(X_val_final, y_val, NUM_CLIENTS)

print("\n Particionando dados de TESTE:")
test_partitions = create_federated_partitions(X_test_final, y_test, NUM_CLIENTS)

print("\n Particionamento concluído!")

In [ ]:
# Visualização da distribuição de classes por cliente
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
datasets = [('Treino', train_partitions), ('Validação', val_partitions), ('Teste', test_partitions)]

for idx, (name, partitions) in enumerate(datasets):
    fraud_counts = [np.sum(partitions[i][1] == 1) for i in range(NUM_CLIENTS)]
    non_fraud_counts = [np.sum(partitions[i][1] == 0) for i in range(NUM_CLIENTS)]
    
    x = np.arange(NUM_CLIENTS)
    width = 0.35
    
    axes[idx].bar(x - width/2, non_fraud_counts, width, label='Não-Fraude', color='skyblue')
    axes[idx].bar(x + width/2, fraud_counts, width, label='Fraude', color='salmon')
    
    axes[idx].set_xlabel('Cliente')
    axes[idx].set_ylabel('Quantidade')
    axes[idx].set_title(f'Distribuição de Classes - {name}')
    axes[idx].set_xticks(x)
    axes[idx].legend()
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Seção 7: Federated Learning - XGBoost

### Configuração:
- **Estratégias**: Bagging (FedAvg) e Cyclic
- **Rounds**: 10 rounds de FL
- **Local Epochs**: 20 boosting rounds por cliente
- **Agregação**: Média ponderada dos modelos

In [ ]:
# Configuracao XGBoost (usando hiperparametros otimizados)
xgb_params = xgb_best_params.copy()
xgb_params.pop('n_estimators', None)  # Remover n_estimators pois usaremos num_boost_round
xgb_params['seed'] = RANDOM_STATE

NUM_ROUNDS = 10  # Rounds de FL
NUM_LOCAL_ROUNDS = 20  # Boosting rounds por cliente

print("\u2699\ufe0f Configuracao XGBoost FL (hiperparametros otimizados):")
print(f"  - Rounds FL: {NUM_ROUNDS}")
print(f"  - Local boosting rounds: {NUM_LOCAL_ROUNDS}")
print(f"  - Clientes: {NUM_CLIENTS}")
print(f"\n  Hiperparametros otimizados:")
for key, value in xgb_params.items():
    print(f"    {key}: {value}")

In [ ]:
class XGBoostClient(fl.client.NumPyClient):
    """
    Cliente Flower para XGBoost.
    Simples, elegante e fácil de entender.
    """
    
    def __init__(self, cid: int, X_train, y_train, X_val, y_val):
        self.cid = cid
        self.X_train = X_train
        self.y_train = y_train
        self.X_val = X_val
        self.y_val = y_val
        
        # Criar DMatrix
        self.dtrain = xgb.DMatrix(X_train, label=y_train)
        self.dval = xgb.DMatrix(X_val, label=y_val)
        
        # Modelo local (None no início)
        self.model = None
    
    def get_parameters(self, config):
        """Retorna parâmetros do modelo local."""
        if self.model is None:
            # Primeira rodada: retornar modelo vazio
            return []
        
        # Serializar modelo
        model_bytes = self.model.save_raw('json')
        return [np.frombuffer(model_bytes, dtype=np.uint8)]
    
    def set_parameters(self, parameters):
        """Atualiza modelo local com parâmetros do servidor."""
        if len(parameters) == 0:
            # Primeira rodada: modelo vazio
            self.model = None
            return
        
        # Deserializar modelo
        model_bytes = parameters[0].tobytes()
        self.model = xgb.Booster()
        self.model.load_model(bytearray(model_bytes))
    
    def fit(self, parameters, config):
        """Treina modelo local."""
        print(f"\n[Cliente {self.cid}] Round {config.get('round', '?')} - Treinando...")
        
        # Atualizar com modelo global
        self.set_parameters(parameters)
        
        # Treinar localmente
        self.model = xgb.train(
            xgb_params,
            self.dtrain,
            num_boost_round=NUM_LOCAL_ROUNDS,
            xgb_model=self.model,  # Continuar do modelo global
            evals=[(self.dtrain, 'train'), (self.dval, 'val')],
            verbose_eval=False
        )
        
        # Avaliar no conjunto de treino
        train_pred = self.model.predict(self.dtrain)
        train_auc = roc_auc_score(self.y_train, train_pred)
        train_tpr = calc_tpr_at_fpr(self.y_train, train_pred)
        
        print(f"[Cliente {self.cid}] Treino: AUC={train_auc:.4f} | TPR@5%FPR={train_tpr:.4f}")
        
        # Retornar parâmetros atualizados
        return self.get_parameters({}), len(self.X_train), {
            'train_auc': float(train_auc),
            'train_tpr': float(train_tpr)
        }
    
    def evaluate(self, parameters, config):
        """Avalia modelo no conjunto de validação."""
        # Atualizar com modelo global
        self.set_parameters(parameters)
        
        if self.model is None:
            return 0.0, len(self.X_val), {}
        
        # Predição
        val_pred = self.model.predict(self.dval)
        val_auc = roc_auc_score(self.y_val, val_pred)
        val_tpr = calc_tpr_at_fpr(self.y_val, val_pred)
        
        print(f"[Cliente {self.cid}] Validação: AUC={val_auc:.4f} | TPR@5%FPR={val_tpr:.4f}")
        
        return float(val_auc), len(self.X_val), {
            'val_auc': float(val_auc),
            'val_tpr': float(val_tpr)
        }


def client_fn_xgb(cid: str) -> XGBoostClient:
    """Factory para criar clientes XGBoost."""
    client_id = int(cid)
    X_train, y_train = train_partitions[client_id]
    X_val, y_val = val_partitions[client_id]
    return XGBoostClient(client_id, X_train, y_train, X_val, y_val)


print(" Cliente XGBoost definido!")

In [ ]:
# Estratégia FedAvg (Bagging)
print("\n Executando XGBoost FL - Estratégia BAGGING (FedAvg)...\n")

xgb_bagging_history = {'rounds': [], 'train_metrics': [], 'val_metrics': []}

strategy_bagging = FedAvg(
    fraction_fit=1.0,  # Todos os clientes participam
    fraction_evaluate=1.0,
    min_fit_clients=NUM_CLIENTS,
    min_evaluate_clients=NUM_CLIENTS,
    min_available_clients=NUM_CLIENTS
)

# Executar simulação
xgb_bagging_result = start_simulation(
    client_fn=client_fn_xgb,
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=strategy_bagging,
    client_resources={'num_cpus': 1}
)

print("\n XGBoost Bagging concluído!")

In [ ]:
# Estratégia Cyclic (1 cliente por round)
print("\n Executando XGBoost FL - Estratégia CYCLIC...\n")

class CyclicStrategy(FedAvg):
    """Estratégia que seleciona 1 cliente por round de forma cíclica."""
    
    def __init__(self, num_clients: int, **kwargs):
        super().__init__(**kwargs)
        self.num_clients = num_clients
        self.current_round = 0
    
    def configure_fit(self, server_round, parameters, client_manager):
        """Seleciona apenas 1 cliente por round (cíclico)."""
        self.current_round = server_round
        
        # Cliente selecionado (round % num_clients)
        selected_client_id = (server_round - 1) % self.num_clients
        
        # Obter todos os clientes disponíveis
        all_clients = list(client_manager.all().values())
        
        # Selecionar apenas o cliente do round atual
        selected_client = [all_clients[selected_client_id]]
        
        print(f"\n[Round {server_round}] Selecionado: Cliente {selected_client_id}")
        
        # Configuração para o cliente
        config = {'round': server_round}
        fit_ins = fl.common.FitIns(parameters, config)
        
        return [(client, fit_ins) for client in selected_client]


xgb_cyclic_history = {'rounds': [], 'train_metrics': [], 'val_metrics': []}

strategy_cyclic = CyclicStrategy(
    num_clients=NUM_CLIENTS,
    fraction_fit=1.0,
    fraction_evaluate=1.0,
    min_fit_clients=1,  # 1 cliente por round
    min_evaluate_clients=NUM_CLIENTS,
    min_available_clients=NUM_CLIENTS
)

# Executar simulação
xgb_cyclic_result = start_simulation(
    client_fn=client_fn_xgb,
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=strategy_cyclic,
    client_resources={'num_cpus': 1}
)

print("\n XGBoost Cyclic concluído!")

---
## Seção 8: Federated Learning - LightGBM

### Mesma estrutura do XGBoost, adaptada para LightGBM

In [ ]:
# Configuracao LightGBM (usando hiperparametros otimizados)
lgb_params = lgb_best_params.copy()
lgb_params.pop('n_estimators', None)  # Remover n_estimators pois usaremos num_boost_round
lgb_params['seed'] = RANDOM_STATE

print("\u2699\ufe0f Configuracao LightGBM FL (hiperparametros otimizados):")
print(f"  - Rounds FL: {NUM_ROUNDS}")
print(f"  - Local boosting rounds: {NUM_LOCAL_ROUNDS}")
print(f"  - Clientes: {NUM_CLIENTS}")
print(f"\n  Hiperparametros otimizados:")
for key, value in lgb_params.items():
    print(f"    {key}: {value}")

In [ ]:
import tempfile
import os

class LightGBMClient(fl.client.NumPyClient):
    """
    Cliente Flower para LightGBM.
    """
    
    def __init__(self, cid: int, X_train, y_train, X_val, y_val):
        self.cid = cid
        self.X_train = X_train
        self.y_train = y_train
        self.X_val = X_val
        self.y_val = y_val
        
        # Criar Datasets
        self.train_data = lgb.Dataset(X_train, label=y_train)
        self.val_data = lgb.Dataset(X_val, label=y_val, reference=self.train_data)
        
        self.model = None
    
    def get_parameters(self, config):
        if self.model is None:
            return []
        
        # Salvar modelo em arquivo temporário
        with tempfile.NamedTemporaryFile(delete=False, suffix='.txt') as tmp:
            self.model.save_model(tmp.name)
            with open(tmp.name, 'rb') as f:
                model_bytes = f.read()
            os.unlink(tmp.name)
        
        return [np.frombuffer(model_bytes, dtype=np.uint8)]
    
    def set_parameters(self, parameters):
        if len(parameters) == 0:
            self.model = None
            return
        
        # Carregar de arquivo temporário
        model_bytes = parameters[0].tobytes()
        with tempfile.NamedTemporaryFile(delete=False, suffix='.txt') as tmp:
            tmp.write(model_bytes)
            tmp.flush()
            self.model = lgb.Booster(model_file=tmp.name)
            os.unlink(tmp.name)
    
    def fit(self, parameters, config):
        print(f"\n[Cliente {self.cid}] Round {config.get('round', '?')} - Treinando...")
        
        self.set_parameters(parameters)
        
        # Treinar
        self.model = lgb.train(
            lgb_params,
            self.train_data,
            num_boost_round=NUM_LOCAL_ROUNDS,
            init_model=self.model,
            valid_sets=[self.train_data, self.val_data],
            valid_names=['train', 'val'],
            callbacks=[lgb.log_evaluation(period=0)]  # Sem verbose
        )
        
        # Avaliar
        train_pred = self.model.predict(self.X_train)
        train_auc = roc_auc_score(self.y_train, train_pred)
        train_tpr = calc_tpr_at_fpr(self.y_train, train_pred)
        
        print(f"[Cliente {self.cid}] Treino: AUC={train_auc:.4f} | TPR@5%FPR={train_tpr:.4f}")
        
        return self.get_parameters({}), len(self.X_train), {
            'train_auc': float(train_auc),
            'train_tpr': float(train_tpr)
        }
    
    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        
        if self.model is None:
            return 0.0, len(self.X_val), {}
        
        val_pred = self.model.predict(self.X_val)
        val_auc = roc_auc_score(self.y_val, val_pred)
        val_tpr = calc_tpr_at_fpr(self.y_val, val_pred)
        
        print(f"[Cliente {self.cid}] Validação: AUC={val_auc:.4f} | TPR@5%FPR={val_tpr:.4f}")
        
        return float(val_auc), len(self.X_val), {
            'val_auc': float(val_auc),
            'val_tpr': float(val_tpr)
        }


def client_fn_lgb(cid: str) -> LightGBMClient:
    client_id = int(cid)
    X_train, y_train = train_partitions[client_id]
    X_val, y_val = val_partitions[client_id]
    return LightGBMClient(client_id, X_train, y_train, X_val, y_val)


print(" Cliente LightGBM definido!")

In [ ]:
# LightGBM Bagging
print("\n Executando LightGBM FL - Estratégia BAGGING...\n")

lgb_bagging_result = start_simulation(
    client_fn=client_fn_lgb,
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=FedAvg(
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS
    ),
    client_resources={'num_cpus': 1}
)

print("\n LightGBM Bagging concluído!")

In [ ]:
# LightGBM Cyclic
print("\n Executando LightGBM FL - Estratégia CYCLIC...\n")

lgb_cyclic_result = start_simulation(
    client_fn=client_fn_lgb,
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=CyclicStrategy(
        num_clients=NUM_CLIENTS,
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=1,
        min_evaluate_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS
    ),
    client_resources={'num_cpus': 1}
)

print("\n LightGBM Cyclic concluído!")

---
## Seção 9: Federated Learning - CatBoost

### Mesma estrutura, adaptada para CatBoost

In [ ]:
# Configuracao CatBoost (usando hiperparametros otimizados)
catboost_params = catboost_best_params.copy()
catboost_params.pop('iterations', None)  # Remover iterations pois usaremos num_boost_round
catboost_params['random_seed'] = RANDOM_STATE

print("\u2699\ufe0f Configuracao CatBoost FL (hiperparametros otimizados):")
print(f"  - Rounds FL: {NUM_ROUNDS}")
print(f"  - Local boosting rounds: {NUM_LOCAL_ROUNDS}")
print(f"  - Clientes: {NUM_CLIENTS}")
print(f"\n  Hiperparametros otimizados:")
for key, value in catboost_params.items():
    print(f"    {key}: {value}")

In [ ]:
class CatBoostClient(fl.client.NumPyClient):
    """
    Cliente Flower para CatBoost.
    """
    
    def __init__(self, cid: int, X_train, y_train, X_val, y_val):
        self.cid = cid
        self.X_train = X_train
        self.y_train = y_train
        self.X_val = X_val
        self.y_val = y_val
        
        # Criar Pools
        self.train_pool = Pool(X_train, label=y_train)
        self.val_pool = Pool(X_val, label=y_val)
        
        self.model = None
    
    def get_parameters(self, config):
        if self.model is None:
            return []
        
        # Salvar em arquivo temporário
        with tempfile.NamedTemporaryFile(delete=False, suffix='.cbm') as tmp:
            self.model.save_model(tmp.name, format='cbm')
            with open(tmp.name, 'rb') as f:
                model_bytes = f.read()
            os.unlink(tmp.name)
        
        return [np.frombuffer(model_bytes, dtype=np.uint8)]
    
    def set_parameters(self, parameters):
        if len(parameters) == 0:
            self.model = None
            return
        
        model_bytes = parameters[0].tobytes()
        with tempfile.NamedTemporaryFile(delete=False, suffix='.cbm') as tmp:
            tmp.write(model_bytes)
            tmp.flush()
            self.model = CatBoostClassifier()
            self.model.load_model(tmp.name)
            os.unlink(tmp.name)
    
    def fit(self, parameters, config):
        print(f"\n[Cliente {self.cid}] Round {config.get('round', '?')} - Treinando...")
        
        self.set_parameters(parameters)
        
        # Criar novo modelo se necessário
        if self.model is None:
            self.model = CatBoostClassifier(**catboost_params)
        
        # Treinar
        self.model.fit(
            self.train_pool,
            eval_set=self.val_pool,
            iterations=NUM_LOCAL_ROUNDS,
            verbose=False,
            init_model=self.model if parameters else None
        )
        
        # Avaliar
        train_pred = self.model.predict_proba(self.X_train)[:, 1]
        train_auc = roc_auc_score(self.y_train, train_pred)
        train_tpr = calc_tpr_at_fpr(self.y_train, train_pred)
        
        print(f"[Cliente {self.cid}] Treino: AUC={train_auc:.4f} | TPR@5%FPR={train_tpr:.4f}")
        
        return self.get_parameters({}), len(self.X_train), {
            'train_auc': float(train_auc),
            'train_tpr': float(train_tpr)
        }
    
    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        
        if self.model is None:
            return 0.0, len(self.X_val), {}
        
        val_pred = self.model.predict_proba(self.X_val)[:, 1]
        val_auc = roc_auc_score(self.y_val, val_pred)
        val_tpr = calc_tpr_at_fpr(self.y_val, val_pred)
        
        print(f"[Cliente {self.cid}] Validação: AUC={val_auc:.4f} | TPR@5%FPR={val_tpr:.4f}")
        
        return float(val_auc), len(self.X_val), {
            'val_auc': float(val_auc),
            'val_tpr': float(val_tpr)
        }


def client_fn_catboost(cid: str) -> CatBoostClient:
    client_id = int(cid)
    X_train, y_train = train_partitions[client_id]
    X_val, y_val = val_partitions[client_id]
    return CatBoostClient(client_id, X_train, y_train, X_val, y_val)


print(" Cliente CatBoost definido!")

In [ ]:
# CatBoost Bagging
print("\n Executando CatBoost FL - Estratégia BAGGING...\n")

catboost_bagging_result = start_simulation(
    client_fn=client_fn_catboost,
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=FedAvg(
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS
    ),
    client_resources={'num_cpus': 1}
)

print("\n CatBoost Bagging concluído!")

In [ ]:
# CatBoost Cyclic
print("\n Executando CatBoost FL - Estratégia CYCLIC...\n")

catboost_cyclic_result = start_simulation(
    client_fn=client_fn_catboost,
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=CyclicStrategy(
        num_clients=NUM_CLIENTS,
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=1,
        min_evaluate_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS
    ),
    client_resources={'num_cpus': 1}
)

print("\n CatBoost Cyclic concluído!")

---
## Seção 10: Visualizações e Comparação de Resultados

### Plots:
1. Convergência de métricas por round (AUC e TPR@5%FPR)
2. Comparação entre estratégias (Bagging vs Cyclic)
3. Comparação entre modelos (XGBoost, LightGBM, CatBoost)
4. Avaliação final no conjunto de teste

In [ ]:
def extract_metrics_from_history(history):
    """
    Extrai métricas do histórico do Flower.
    """
    rounds = []
    losses = []
    
    if hasattr(history, 'losses_distributed'):
        for round_num, loss_val in history.losses_distributed:
            rounds.append(round_num)
            losses.append(loss_val)
    
    return rounds, losses


# Extrair métricas
print(" Extraindo métricas dos experimentos FL...")

# XGBoost
xgb_bagging_rounds, xgb_bagging_losses = extract_metrics_from_history(xgb_bagging_result.history)
xgb_cyclic_rounds, xgb_cyclic_losses = extract_metrics_from_history(xgb_cyclic_result.history)

# LightGBM
lgb_bagging_rounds, lgb_bagging_losses = extract_metrics_from_history(lgb_bagging_result.history)
lgb_cyclic_rounds, lgb_cyclic_losses = extract_metrics_from_history(lgb_cyclic_result.history)

# CatBoost
catboost_bagging_rounds, catboost_bagging_losses = extract_metrics_from_history(catboost_bagging_result.history)
catboost_cyclic_rounds, catboost_cyclic_losses = extract_metrics_from_history(catboost_cyclic_result.history)

print(" Métricas extraídas!")

In [ ]:
# Plot 1: Comparação de Convergência
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Convergência de Métricas - Federated Learning', fontsize=16, fontweight='bold')

models = [
    ('XGBoost', xgb_bagging_rounds, xgb_bagging_losses, xgb_cyclic_rounds, xgb_cyclic_losses),
    ('LightGBM', lgb_bagging_rounds, lgb_bagging_losses, lgb_cyclic_rounds, lgb_cyclic_losses),
    ('CatBoost', catboost_bagging_rounds, catboost_bagging_losses, catboost_cyclic_rounds, catboost_cyclic_losses)
]

for idx, (name, bag_rounds, bag_losses, cyc_rounds, cyc_losses) in enumerate(models):
    # Loss
    axes[0, idx].plot(bag_rounds, bag_losses, marker='o', label='Bagging', linewidth=2)
    axes[0, idx].plot(cyc_rounds, cyc_losses, marker='s', label='Cyclic', linewidth=2)
    axes[0, idx].set_title(f'{name} - Loss', fontweight='bold')
    axes[0, idx].set_xlabel('Round')
    axes[0, idx].set_ylabel('Loss')
    axes[0, idx].legend()
    axes[0, idx].grid(alpha=0.3)
    
    # Placeholder para métricas (se disponível)
    axes[1, idx].text(0.5, 0.5, 'Métricas por round\n(AUC/TPR)', 
                      ha='center', va='center', fontsize=12)
    axes[1, idx].set_title(f'{name} - Métricas', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Comparação entre Modelos e Estratégias
fig, ax = plt.subplots(figsize=(12, 6))

experiments = [
    'XGBoost\nBagging', 'XGBoost\nCyclic',
    'LightGBM\nBagging', 'LightGBM\nCyclic',
    'CatBoost\nBagging', 'CatBoost\nCyclic'
]

# Placeholder: loss final de cada experimento
final_losses = [
    xgb_bagging_losses[-1] if xgb_bagging_losses else 0,
    xgb_cyclic_losses[-1] if xgb_cyclic_losses else 0,
    lgb_bagging_losses[-1] if lgb_bagging_losses else 0,
    lgb_cyclic_losses[-1] if lgb_cyclic_losses else 0,
    catboost_bagging_losses[-1] if catboost_bagging_losses else 0,
    catboost_cyclic_losses[-1] if catboost_cyclic_losses else 0
]

colors = ['skyblue', 'lightblue', 'lightgreen', 'palegreen', 'salmon', 'lightsalmon']
bars = ax.bar(experiments, final_losses, color=colors, edgecolor='black', linewidth=1.5)

ax.set_ylabel('Loss Final', fontsize=12, fontweight='bold')
ax.set_title('Comparação de Loss Final - Todos os Experimentos FL', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Adicionar valores nas barras
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}',
            ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "="*80)
print("RESUMO DOS EXPERIMENTOS FEDERATED LEARNING")
print("="*80)

print("\n Experimentos concluídos com sucesso!")
print(f"\n Configuração:")
print(f"  - Número de clientes: {NUM_CLIENTS}")
print(f"  - Rounds FL: {NUM_ROUNDS}")
print(f"  - Local boosting rounds: {NUM_LOCAL_ROUNDS}")
print(f"  - Dataset: BAF Base")
print(f"  - Modelos: XGBoost, LightGBM, CatBoost")
print(f"  - Estratégias: Bagging (FedAvg) e Cyclic")

print("\n Resultados Finais:")
print(f"\n  XGBoost:")
print(f"    - Bagging Loss: {xgb_bagging_losses[-1]:.4f}" if xgb_bagging_losses else "    - Bagging: N/A")
print(f"    - Cyclic Loss: {xgb_cyclic_losses[-1]:.4f}" if xgb_cyclic_losses else "    - Cyclic: N/A")

print(f"\n  LightGBM:")
print(f"    - Bagging Loss: {lgb_bagging_losses[-1]:.4f}" if lgb_bagging_losses else "    - Bagging: N/A")
print(f"    - Cyclic Loss: {lgb_cyclic_losses[-1]:.4f}" if lgb_cyclic_losses else "    - Cyclic: N/A")

print(f"\n  CatBoost:")
print(f"    - Bagging Loss: {catboost_bagging_losses[-1]:.4f}" if catboost_bagging_losses else "    - Bagging: N/A")
print(f"    - Cyclic Loss: {catboost_cyclic_losses[-1]:.4f}" if catboost_cyclic_losses else "    - Cyclic: N/A")

print("\n" + "="*80)

---
## Conclusão

Este notebook implementa **Federated Learning** para detecção de fraude bancária, mantendo:

 **100% do preprocessamento** do notebook original  
 **100% das estratégias** de feature engineering  
 **100% dos modelos** (XGBoost, LightGBM, CatBoost)  
 **Particionamento balanceado** para 3 clientes  
 **Estratégias FL**: Bagging e Cyclic  
 **Código simples, elegante e fácil de entender**  

### Próximos Passos:
- Avaliar modelos agregados no conjunto de teste (Mês 7)
- Comparar com resultados centralizados (notebook original)
- Análise de fairness no contexto federado
- Experimentar com mais clientes e diferentes particionamentos

---
## Seção 11: Análise de Fairness no Federated Learning

### Objetivo:
Avaliar se os modelos federalizados apresentam viés contra clientes com idade > 50 anos.

### Métricas de Fairness:
- **FPR (False Positive Rate)**: Taxa de clientes legítimos marcados como fraude
- **Fairness Ratio**: FPR_old / FPR_young (ideal = 1.0, sem viés)
- **Meta**: Fairness Ratio próximo de 1.0 (sem discriminação)

### Estratégias de Mitigação:
1. **Threshold Decoupling**: Thresholds diferentes por grupo etário
2. **Sample Weighting**: Aumentar peso de clientes legítimos > 50 anos no treino
3. **Combinação**: Aplicar ambas as estratégias

In [ ]:
# ============================================================================
# FUNÇÕES AUXILIARES PARA ANÁLISE DE FAIRNESS
# ============================================================================

def calc_fairness_metrics(y_true, y_prob, age_flag, fpr_target=0.05):
    """
    Calcula métricas de fairness para um modelo.
    
    Args:
        y_true: Labels verdadeiros (0=legítimo, 1=fraude)
        y_prob: Probabilidades preditas
        age_flag: Flag binária (0=<=50 anos, 1=>50 anos)
        fpr_target: FPR alvo para threshold global
    
    Returns:
        dict: Métricas de fairness
    """
    # Threshold global
    threshold = get_threshold_at_fpr(y_true, y_prob, fpr_target)
    y_pred = (y_prob >= threshold).astype(int)
    
    # Máscaras de idade
    mask_old = age_flag == 1      # Idade > 50
    mask_young = age_flag == 0    # Idade <= 50
    
    # FPR por grupo
    def calc_fpr(y_true_g, y_pred_g):
        neg_mask = y_true_g == 0
        if neg_mask.sum() == 0:
            return 0.0
        fp = ((y_pred_g == 1) & (y_true_g == 0)).sum()
        return fp / neg_mask.sum()
    
    fpr_old = calc_fpr(y_true[mask_old], y_pred[mask_old])
    fpr_young = calc_fpr(y_true[mask_young], y_pred[mask_young])
    
    fairness_ratio = fpr_old / fpr_young if fpr_young > 0 else np.inf
    
    # Métricas gerais
    auc = roc_auc_score(y_true, y_prob)
    tpr = calc_tpr_at_fpr(y_true, y_prob, fpr_target)
    
    return {
        'threshold': threshold,
        'auc': auc,
        'tpr_at_5fpr': tpr,
        'fpr_old': fpr_old,
        'fpr_young': fpr_young,
        'fairness_ratio': fairness_ratio
    }


def get_group_threshold_at_fpr(y_true, y_prob, group_mask, fpr_target=0.05):
    """
    Calcula threshold específico para um grupo que resulta em FPR <= fpr_target.
    """
    y_true_g = y_true[group_mask]
    y_prob_g = y_prob[group_mask]
    
    if len(y_true_g) == 0:
        return 0.5
    
    fpr, tpr, thresholds = roc_curve(y_true_g, y_prob_g)
    valid_indices = np.where(fpr <= fpr_target)[0]
    
    if len(valid_indices) == 0:
        return 1.0
    
    best_idx = valid_indices[np.argmax(tpr[valid_indices])]
    return thresholds[best_idx]


def apply_decoupled_thresholds(y_true, y_prob, age_flag, fpr_target=0.05):
    """
    Aplica thresholds diferentes para grupos etários.
    """
    mask_old = age_flag == 1
    mask_young = age_flag == 0
    
    # Calcular thresholds por grupo
    threshold_old = get_group_threshold_at_fpr(y_true, y_prob, mask_old, fpr_target)
    threshold_young = get_group_threshold_at_fpr(y_true, y_prob, mask_young, fpr_target)
    
    # Aplicar thresholds
    y_pred = np.zeros(len(y_true), dtype=int)
    y_pred[mask_old] = (y_prob[mask_old] >= threshold_old).astype(int)
    y_pred[mask_young] = (y_prob[mask_young] >= threshold_young).astype(int)
    
    return y_pred, threshold_old, threshold_young


print("Funções de fairness definidas!")

---
### 11.1 - Avaliação de Fairness dos Modelos FL

Primeiro, vamos recuperar os modelos globais finais de cada experimento FL e avaliar suas métricas de fairness no conjunto de teste.

In [ ]:
# ============================================================================
# PREPARAR DADOS DE TESTE PARA ANÁLISE DE FAIRNESS
# ============================================================================

print("Combinando dados de teste de todos os clientes...")

# Combinar X_test e y_test de todos os clientes
X_test_all = np.vstack([test_partitions[i][0] for i in range(NUM_CLIENTS)])
y_test_all = np.concatenate([test_partitions[i][1] for i in range(NUM_CLIENTS)])

print(f"X_test_all shape: {X_test_all.shape}")
print(f"y_test_all shape: {y_test_all.shape}")

# IMPORTANTE: Recuperar age_above_50 para o conjunto de teste
# Usamos age_above_50_test_full que foi salvo antes do particionamento
age_above_50_test_all = age_above_50_test_full.values

print(f"age_above_50_test_all shape: {age_above_50_test_all.shape}")
print(f"\nDistribuição etária no teste:")
print(f"  <= 50 anos: {(age_above_50_test_all == 0).sum()} ({(age_above_50_test_all == 0).mean()*100:.1f}%)")
print(f"  > 50 anos: {(age_above_50_test_all == 1).sum()} ({(age_above_50_test_all == 1).mean()*100:.1f}%)")

# Verificar taxa de fraude por grupo etário
mask_old = age_above_50_test_all == 1
mask_young = age_above_50_test_all == 0

fraud_rate_old = y_test_all[mask_old].mean()
fraud_rate_young = y_test_all[mask_young].mean()

print(f"\nTaxa de fraude por grupo:")
print(f"  <= 50 anos: {fraud_rate_young*100:.2f}%")
print(f"  > 50 anos: {fraud_rate_old*100:.2f}%")

**NOTA IMPORTANTE**: Para avaliar os modelos FL, precisaríamos:

1. Salvar os modelos globais finais durante a simulação FL
2. Carregar esses modelos e fazer predições no X_test_all
3. Calcular as métricas de fairness

Como a implementação atual do Flower não retorna diretamente o modelo global final da simulação, aqui está o código para **quando você implementar a salvação dos modelos**:

```python
# Exemplo de como avaliar (quando tiver os modelos salvos)
# XGBoost Bagging
y_prob_xgb_bagging = xgb_bagging_model.predict(xgb.DMatrix(X_test_all))
fairness_xgb_bagging = calc_fairness_metrics(y_test_all, y_prob_xgb_bagging, age_above_50_test_all)

# XGBoost Cyclic
y_prob_xgb_cyclic = xgb_cyclic_model.predict(xgb.DMatrix(X_test_all))
fairness_xgb_cyclic = calc_fairness_metrics(y_test_all, y_prob_xgb_cyclic, age_above_50_test_all)

# ... e assim por diante para LightGBM e CatBoost
```

Por enquanto, vamos demonstrar as estratégias de mitigação de fairness usando um modelo de exemplo.

In [ ]:
# ============================================================================
# DEMONSTRAÇÃO: ANÁLISE DE FAIRNESS COM MODELO DE EXEMPLO
# ============================================================================

print("DEMONSTRAÇÃO: Treinando modelo XGBoost local para análise de fairness...")
print("(Em produção, você usaria os modelos globais salvos do FL)\n")

# Treinar modelo local para demonstração
dtrain_demo = xgb.DMatrix(X_train_final, label=y_train.values)
dtest_demo = xgb.DMatrix(X_test_all, label=y_test_all)

demo_model = xgb.train(
    xgb_params,
    dtrain_demo,
    num_boost_round=100,
    verbose_eval=False
)

# Predições no teste
y_prob_demo = demo_model.predict(dtest_demo)

# Avaliar fairness do modelo baseline
print("=" * 70)
print("MODELO BASELINE (SEM MITIGAÇÃO DE FAIRNESS)")
print("=" * 70)

fairness_baseline = calc_fairness_metrics(y_test_all, y_prob_demo, age_above_50_test_all)

print(f"\nMétricas Gerais:")
print(f"  AUC: {fairness_baseline['auc']:.4f}")
print(f"  TPR @ 5% FPR: {fairness_baseline['tpr_at_5fpr']:.4f}")
print(f"  Threshold: {fairness_baseline['threshold']:.4f}")

print(f"\nMétricas de Fairness:")
print(f"  FPR (> 50 anos): {fairness_baseline['fpr_old']*100:.2f}%")
print(f"  FPR (<= 50 anos): {fairness_baseline['fpr_young']*100:.2f}%")
print(f"  Fairness Ratio: {fairness_baseline['fairness_ratio']:.2f}")

if fairness_baseline['fairness_ratio'] > 1.2:
    print("\n  VIÉS DETECTADO: Modelo discrimina clientes > 50 anos!")
elif fairness_baseline['fairness_ratio'] < 0.8:
    print("\n  VIÉS DETECTADO: Modelo discrimina clientes <= 50 anos!")
else:
    print("\n  OK: Fairness ratio aceitável (próximo de 1.0)")

---
### 11.2 - Estratégia A: Threshold Decoupling

Aplicar thresholds diferentes para cada grupo etário, mantendo FPR = 5% **dentro de cada grupo**.

In [ ]:
# ============================================================================
# ESTRATÉGIA A: THRESHOLD DECOUPLING
# ============================================================================

print("=" * 70)
print("ESTRATÉGIA A: THRESHOLD DECOUPLING")
print("=" * 70)

# Aplicar thresholds decoupled
y_pred_decoupled, threshold_old, threshold_young = apply_decoupled_thresholds(
    y_test_all, y_prob_demo, age_above_50_test_all, fpr_target=0.05
)

print(f"\nThresholds por grupo:")
print(f"  > 50 anos: {threshold_old:.4f}")
print(f"  <= 50 anos: {threshold_young:.4f}")

# Calcular métricas com thresholds decoupled
mask_old = age_above_50_test_all == 1
mask_young = age_above_50_test_all == 0

def calc_metrics_with_pred(y_true, y_pred, mask):
    y_t = y_true[mask]
    y_p = y_pred[mask]
    
    # FPR
    neg_mask = y_t == 0
    fpr = ((y_p == 1) & (y_t == 0)).sum() / neg_mask.sum() if neg_mask.sum() > 0 else 0
    
    # TPR
    pos_mask = y_t == 1
    tpr = ((y_p == 1) & (y_t == 1)).sum() / pos_mask.sum() if pos_mask.sum() > 0 else 0
    
    return {'fpr': fpr, 'tpr': tpr}

metrics_old_decoupled = calc_metrics_with_pred(y_test_all, y_pred_decoupled, mask_old)
metrics_young_decoupled = calc_metrics_with_pred(y_test_all, y_pred_decoupled, mask_young)

fairness_ratio_decoupled = metrics_old_decoupled['fpr'] / metrics_young_decoupled['fpr'] if metrics_young_decoupled['fpr'] > 0 else np.inf

print(f"\nMétricas após Threshold Decoupling:")
print(f"  FPR (> 50 anos): {metrics_old_decoupled['fpr']*100:.2f}%")
print(f"  FPR (<= 50 anos): {metrics_young_decoupled['fpr']*100:.2f}%")
print(f"  Fairness Ratio: {fairness_ratio_decoupled:.2f}")
print(f"\n  TPR (> 50 anos): {metrics_old_decoupled['tpr']*100:.2f}%")
print(f"  TPR (<= 50 anos): {metrics_young_decoupled['tpr']*100:.2f}%")

# Overall TPR
overall_tpr_decoupled = (y_pred_decoupled[y_test_all == 1] == 1).sum() / (y_test_all == 1).sum()
print(f"\n  TPR Geral: {overall_tpr_decoupled*100:.2f}%")

---
### 11.3 - Estratégia B: Sample Weighting

Re-treinar o modelo aumentando o peso de clientes legítimos (classe 0) com idade > 50 anos.

In [ ]:
# ============================================================================
# ESTRATÉGIA B: SAMPLE WEIGHTING
# ============================================================================

print("=" * 70)
print("ESTRATÉGIA B: SAMPLE WEIGHTING")
print("=" * 70)

# Criar sample weights para o treino
# IMPORTANTE: Usamos age_above_50_train_full salvo anteriormente
sample_weights_train = np.ones(len(y_train))

# Aumentar peso de clientes LEGÍTIMOS (y=0) com idade > 50
WEIGHT_OLD_LEGIT = 3.0  # Peso 3x maior

age_above_50_train = age_above_50_train_full.values
mask_old_legit = (y_train.values == 0) & (age_above_50_train == 1)

sample_weights_train[mask_old_legit] = WEIGHT_OLD_LEGIT

print(f"\nSample weights criados:")
print(f"  Amostras com peso 1.0: {(sample_weights_train == 1.0).sum()}")
print(f"  Amostras com peso {WEIGHT_OLD_LEGIT}: {(sample_weights_train == WEIGHT_OLD_LEGIT).sum()}")
print(f"    (clientes legítimos > 50 anos)")

# Re-treinar modelo com sample weights
print(f"\nRe-treinando modelo XGBoost com sample weighting...")

dtrain_weighted = xgb.DMatrix(X_train_final, label=y_train.values, weight=sample_weights_train)

model_weighted = xgb.train(
    xgb_params,
    dtrain_weighted,
    num_boost_round=100,
    verbose_eval=False
)

# Predições no teste
y_prob_weighted = model_weighted.predict(dtest_demo)

# Avaliar fairness
fairness_weighted = calc_fairness_metrics(y_test_all, y_prob_weighted, age_above_50_test_all)

print(f"\nMétricas Gerais (com sample weighting):")
print(f"  AUC: {fairness_weighted['auc']:.4f}")
print(f"  TPR @ 5% FPR: {fairness_weighted['tpr_at_5fpr']:.4f}")
print(f"  Threshold: {fairness_weighted['threshold']:.4f}")

print(f"\nMétricas de Fairness (com sample weighting):")
print(f"  FPR (> 50 anos): {fairness_weighted['fpr_old']*100:.2f}%")
print(f"  FPR (<= 50 anos): {fairness_weighted['fpr_young']*100:.2f}%")
print(f"  Fairness Ratio: {fairness_weighted['fairness_ratio']:.2f}")

if fairness_weighted['fairness_ratio'] > 1.2:
    print("\n  VIÉS DETECTADO: Modelo ainda discrimina clientes > 50 anos")
elif fairness_weighted['fairness_ratio'] < 0.8:
    print("\n  VIÉS DETECTADO: Modelo agora discrimina clientes <= 50 anos")
else:
    print("\n  MELHORIA: Fairness ratio melhorou!")

---
### 11.4 - Comparação de Todas as Estratégias

In [ ]:
# ============================================================================
# COMPARAÇÃO FINAL DE ESTRATÉGIAS
# ============================================================================

print("\n" + "="*70)
print("COMPARAÇÃO DE ESTRATÉGIAS DE FAIRNESS")
print("="*70)

# Criar tabela comparativa
comparison_data = {
    'Estratégia': [
        'Baseline (sem mitigação)',
        'Threshold Decoupling',
        'Sample Weighting'
    ],
    'AUC': [
        fairness_baseline['auc'],
        roc_auc_score(y_test_all, y_prob_demo),  # Mesmo modelo, só muda threshold
        fairness_weighted['auc']
    ],
    'TPR@5%FPR': [
        fairness_baseline['tpr_at_5fpr'],
        overall_tpr_decoupled,
        fairness_weighted['tpr_at_5fpr']
    ],
    'FPR >50': [
        fairness_baseline['fpr_old'],
        metrics_old_decoupled['fpr'],
        fairness_weighted['fpr_old']
    ],
    'FPR <=50': [
        fairness_baseline['fpr_young'],
        metrics_young_decoupled['fpr'],
        fairness_weighted['fpr_young']
    ],
    'Fairness Ratio': [
        fairness_baseline['fairness_ratio'],
        fairness_ratio_decoupled,
        fairness_weighted['fairness_ratio']
    ]
}

comparison_df = pd.DataFrame(comparison_data)

print("\n")
print(comparison_df.to_string(index=False))

print("\n" + "="*70)
print("INTERPRETAÇÃO")
print("="*70)
print("\nFairness Ratio ideal = 1.0 (sem discriminação)")
print("Fairness Ratio > 1.2: Viés contra clientes > 50 anos")
print("Fairness Ratio < 0.8: Viés contra clientes <= 50 anos")
print("\nMelhor estratégia: Aquela com Fairness Ratio mais próximo de 1.0")
print("                     mantendo TPR@5%FPR alto")

In [ ]:
# ============================================================================
# VISUALIZAÇÃO: COMPARAÇÃO DE FAIRNESS
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Fairness Ratio
strategies = comparison_df['Estratégia'].values
fairness_ratios = comparison_df['Fairness Ratio'].values

colors = ['red' if r > 1.2 or r < 0.8 else 'green' for r in fairness_ratios]

axes[0].bar(range(len(strategies)), fairness_ratios, color=colors, edgecolor='black', linewidth=1.5)
axes[0].axhline(y=1.0, color='blue', linestyle='--', linewidth=2, label='Ideal (1.0)')
axes[0].axhline(y=1.2, color='orange', linestyle=':', linewidth=1, label='Threshold (1.2)')
axes[0].axhline(y=0.8, color='orange', linestyle=':', linewidth=1)
axes[0].set_xticks(range(len(strategies)))
axes[0].set_xticklabels(strategies, rotation=15, ha='right')
axes[0].set_ylabel('Fairness Ratio', fontweight='bold')
axes[0].set_title('Fairness Ratio por Estratégia', fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: FPR por grupo
x = np.arange(len(strategies))
width = 0.35

fpr_old = comparison_df['FPR >50'].values * 100
fpr_young = comparison_df['FPR <=50'].values * 100

axes[1].bar(x - width/2, fpr_old, width, label='> 50 anos', color='salmon', edgecolor='black')
axes[1].bar(x + width/2, fpr_young, width, label='<= 50 anos', color='skyblue', edgecolor='black')
axes[1].set_xticks(x)
axes[1].set_xticklabels(strategies, rotation=15, ha='right')
axes[1].set_ylabel('FPR (%)', fontweight='bold')
axes[1].set_title('FPR por Grupo Etário', fontweight='bold')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nGráficos gerados com sucesso!")

---
### 11.5 - Conclusões da Análise de Fairness

**Principais Descobertas:**

1. **Baseline**: Modelo sem mitigação provavelmente apresenta viés contra clientes > 50 anos
2. **Threshold Decoupling**: Equaliza FPR entre grupos aplicando thresholds diferentes
3. **Sample Weighting**: Melhora fairness durante o treino, mas pode impactar performance geral

**Recomendações para FL:**

1. **Durante o Treino FL**:
   - Aplicar sample weighting em cada cliente
   - Garantir que cada cliente tenha representação balanceada de grupos etários

2. **Durante a Inferência**:
   - Aplicar threshold decoupling como post-processing
   - Monitorar fairness ratio continuamente

3. **Governança**:
   - Documentar métricas de fairness junto com métricas de performance
   - Estabelecer limites aceitáveis para fairness ratio (ex: 0.9 < ratio < 1.1)
   - Auditorias regulares de fairness

**Próximos Passos:**

- Implementar salvação de modelos globais no FL
- Avaliar fairness de todos os experimentos FL (XGBoost, LightGBM, CatBoost x Bagging/Cyclic)
- Comparar fairness: FL vs. Centralizado
- Investigar se estratégias FL (Bagging vs Cyclic) afetam fairness